<a href="https://colab.research.google.com/github/algulawani/RM-Research-Final-Task/blob/main/face_alignment_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install face-alignment

In [2]:
"""
This script extracts facial landmarks and confidence scores from
all images in the RAF-DB training set using the face-alignment library.

To avoid running out of RAM on Google Colab, images are processed
in batches of 500. For each batch:
1. The face-alignment neural network is loaded into GPU memory
2. Each image in the batch is loaded, resized to 224x224, and
   passed through the detector to get 68 landmarks and scores
3. Results are appended to a running dictionary
4. The detector is deleted and GPU memory is cleared
5. The dictionary is saved to disk as a checkpoint

The dictionary is saved as a .pth file after every batch, so if
Colab disconnects, progress is not lost. On restart, the script
checks which images have already been processed and skips them.

The final output is a .pth file containing a dictionary where:
- Keys are image paths like "1/train_05178_aligned.jpg"
- Values are dictionaries with:
  - "landmarks": numpy array of shape (68, 2)
  - "scores": numpy array of shape (68,)
"""

import os
import gc
import numpy as np
import torch
import face_alignment
from PIL import Image

test_path = "/content/drive/MyDrive/OADN/RAF-DB/2/DATASET/test"
output_dir = "/content/drive/MyDrive/OADN/Preprocessed Testing Data"
output_path = os.path.join(output_dir, "test_landmarks_confidencescores.pth")
os.makedirs(output_dir, exist_ok=True)

BATCH_SIZE = 500

# Collect all image paths
all_images = []
class_folders = sorted(os.listdir(test_path))
for class_folder in class_folders:
    class_path = os.path.join(test_path, class_folder)
    if not os.path.isdir(class_path):
        continue
    for img_name in sorted(os.listdir(class_path)):
        all_images.append(f"{class_folder}/{img_name}")

# Load existing progress if the script was interrupted previously
if os.path.exists(output_path):
    landmarks_dict = torch.load(output_path, weights_only=False)
    print(f"Loaded existing checkpoint with {len(landmarks_dict)} images")
else:
    landmarks_dict = {}

# Filter out already processed images
remaining_images = [img for img in all_images if img not in landmarks_dict]
print(f"Total images: {len(all_images)}")
print(f"Already processed: {len(landmarks_dict)}")
print(f"Remaining: {len(remaining_images)}")

failed_images = []

# Process in batches
for batch_start in range(0, len(remaining_images), BATCH_SIZE):
    batch = remaining_images[batch_start:batch_start + BATCH_SIZE]
    batch_num = (batch_start // BATCH_SIZE) + 1
    total_batches = (len(remaining_images) + BATCH_SIZE - 1) // BATCH_SIZE
    print(f"\nBatch {batch_num}/{total_batches}: processing {len(batch)} images")

    # Load the detector for this batch
    fa = face_alignment.FaceAlignment(
        face_alignment.LandmarksType.TWO_D,
        device='cuda'
    )

    for img_rel_path in batch:
        img_path = os.path.join(test_path, img_rel_path)

        image = Image.open(img_path).convert('RGB')
        image_resized = image.resize((224, 224), Image.BILINEAR)
        image_np = np.array(image_resized)

        try:
            landmarks, scores, _ = fa.get_landmarks_from_image(
                image_np,
                return_landmark_score=True
            )
        except Exception as e:
            failed_images.append(img_rel_path)
            continue

        if landmarks is None or len(landmarks) == 0:
            failed_images.append(img_rel_path)
            continue

        landmarks_dict[img_rel_path] = {
            "landmarks": landmarks[0],
            "scores": scores[0]
        }

    # Delete detector and free GPU memory
    del fa
    gc.collect()
    torch.cuda.empty_cache()

    # Save checkpoint after each batch
    torch.save(landmarks_dict, output_path)
    print(f"Checkpoint saved: {len(landmarks_dict)} images processed so far")

print(f"\nDone. Successfully processed: {len(landmarks_dict)} images")
print(f"Failed/skipped: {len(failed_images)} images")

Total images: 3068
Already processed: 0
Remaining: 3068

Batch 1/7: processing 500 images
Downloading: "https://www.adrianbulat.com/downloads/python-fan/s3fd-619a316812.pth" to /root/.cache/torch/hub/checkpoints/s3fd-619a316812.pth


100%|██████████| 85.7M/85.7M [00:06<00:00, 14.2MB/s]
Downloading: "https://www.adrianbulat.com/downloads/python-fan/2DFAN4-cd938726ad.zip" to /root/.cache/torch/hub/checkpoints/2DFAN4-cd938726ad.zip
100%|██████████| 91.9M/91.9M [00:05<00:00, 16.5MB/s]
/usr/local/lib/python3.12/dist-packages/face_alignment/api.py:147: UserWarning: No faces were detected.
  warnings.warn("No faces were detected.")


Checkpoint saved: 497 images processed so far

Batch 2/7: processing 500 images
Checkpoint saved: 996 images processed so far

Batch 3/7: processing 500 images
Checkpoint saved: 1494 images processed so far

Batch 4/7: processing 500 images
Checkpoint saved: 1991 images processed so far

Batch 5/7: processing 500 images
Checkpoint saved: 2479 images processed so far

Batch 6/7: processing 500 images
Checkpoint saved: 2967 images processed so far

Batch 7/7: processing 68 images
Checkpoint saved: 3034 images processed so far

Done. Successfully processed: 3034 images
Failed/skipped: 34 images
